In [1]:
import duckdb
import os
import pandas as pd
from dotenv import load_dotenv

In [2]:
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
if not os.getenv('HF_TOKEN'):
    load_dotenv('notebooks/.env')
    load_dotenv('.env')
HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Make sure your .env file exists and is formatted correctly.")


In [3]:
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# 3. Map the tables (this doesn't download the data, it just creates pointers)
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':        f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':        f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':         f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':  f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':     f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

# 4. Test the connection instantly using DuckDB SQL
test_query = f"SELECT COUNT(*) FROM {TABLES['fact_daily_sample']}"
row_count = con.sql(test_query).fetchone()[0]

print(f"✅ Connection successful! The sample table has {row_count:,} rows.")

✅ Connection successful! The sample table has 11,694,072 rows.


## 1) The Contract

1. **The Grain:** In my final feature table, one row represents exactly **one specific content item** (URL/page) for a specific client.
2. **The Tables:** I am using `fact_content_daily_performance` (for daily metrics) and `fact_content_query_90d` (for query-level signals).
3. **The Time Window:** I am using a mid-panel slice ending in **March 2026** (features built from Jan 1 to Mar 31; outcome observed in April 2026).
4. **The Label:** A binary flag (`1` or `0`) indicating if the page's impressions **declined by more than 20%** in the following month (April) compared to the current month (March).
5. **The Deliberate Exclusion:** I am excluding any content item that received **fewer than 100 impressions** in the baseline month (March 2026), as ultra-low-traffic pages are too noisy for reliable prediction.

In [4]:
# 2) Verification Queries
# Run this to prove the data matches the contract above.

# Ensure connection and configuration exist if cell is run independently
if 'con' not in globals() or 'REL' not in globals():
    import duckdb, os, pandas as pd
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    if not os.getenv('HF_TOKEN'):
        load_dotenv('notebooks/.env')
        load_dotenv('.env')
    HF_TOKEN = os.getenv('HF_TOKEN')
    REL = 'hf://datasets/FlyRank/internship-warehouse'
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Optimization: Fetch March partition directly into a local temp table once to avoid repeated remote glob scans over HTTP
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE march_daily AS 
    SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

print("--- Query 1: Row Count & Date Span ---")
span_query = """
    SELECT 
        COUNT(*) as total_rows,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM march_daily
"""
display(con.sql(span_query).df())

print("\n--- Query 2: The Grain ---")
# Proving that raw data is daily per content, but our grouped slice will be 1 row per content
grain_query = """
    SELECT client_hash_id, content_hash_id, COUNT(*) as days_active_in_march
    FROM march_daily
    GROUP BY client_hash_id, content_hash_id
    LIMIT 5
"""
display(con.sql(grain_query).df())

print("\n--- Query 3: Availability (Filtered with IS TRUE) ---")
# Proving how many rows survive our deliberate exclusion rule
availability_query = """
    WITH march_totals AS (
        SELECT 
            content_hash_id,
            (SUM(gsc_impressions) >= 100) AS is_eligible
        FROM march_daily
        GROUP BY content_hash_id
    )
    SELECT COUNT(*) as surviving_content_items
    FROM march_totals
    WHERE is_eligible IS TRUE
"""
display(con.sql(availability_query).df())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Query 1: Row Count & Date Span ---


,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31



--- Query 2: The Grain ---


,client_hash_id,content_hash_id,days_active_in_march
0,client_625b6439094e23e4,content_39fa7500959586bb,31
1,client_625b6439094e23e4,content_df68a8e343021338,31
2,client_625b6439094e23e4,content_5ccf881fd4feecae,31
3,client_625b6439094e23e4,content_3fcbe5d32cf03632,31
4,client_625b6439094e23e4,content_be661aaa0ee5830e,31



--- Query 3: Availability (Filtered with IS TRUE) ---


,surviving_content_items
0,101441


In [5]:
## 3) Five Features & The Trap
# We build 5 honest features from the 90 days prior to our decision moment, plus 1 trap.

feature_query = f"""
    WITH base AS (
        SELECT 
            content_hash_id,
            -- Feature 1: Total traffic volume
            SUM(gsc_impressions) AS imp_past_90,
            -- Feature 2: Engagement volume
            SUM(gsc_clicks) AS clicks_past_90,
            -- Feature 3: Ranking quality
            AVG(gsc_avg_position) AS avg_pos_past_90,
            
            -- THE TRAP: Future data leaked into the features!
            SUM(CASE WHEN report_date > '2026-03-31' THEN gsc_impressions ELSE 0 END) as future_impressions,
            
            -- THE LABEL: Did April traffic drop > 20% compared to March?
            CASE 
                WHEN SUM(CASE WHEN report_date > '2026-03-31' AND report_date <= '2026-04-30' THEN gsc_impressions ELSE 0 END) < 
                     (SUM(CASE WHEN report_date >= '2026-03-01' AND report_date <= '2026-03-31' THEN gsc_impressions ELSE 0 END) * 0.8)
                THEN 1 ELSE 0 
            END AS is_declining
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-01-01' AND '2026-04-30'
        GROUP BY content_hash_id
        HAVING SUM(CASE WHEN report_date >= '2026-03-01' AND report_date <= '2026-03-31' THEN gsc_impressions ELSE 0 END) >= 100
    )
    SELECT * FROM base
"""
df_features = con.sql(feature_query).df()

# Add Features 4 and 5 from the query table
qsignals = con.sql(f"SELECT content_hash_id, ANY_VALUE(content_visible_query_count) AS visible_queries, ANY_VALUE(rare_impressions_share) AS rare_share FROM {TABLES['fact_query_90d']} GROUP BY content_hash_id").df()
df_model = df_features.merge(qsignals, on='content_hash_id', how='left').fillna(0)

display(df_model.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,imp_past_90,clicks_past_90,avg_pos_past_90,future_impressions,is_declining,visible_queries,rare_share
0,content_73376814871e4fc2,661.0,0.0,32.856709,24.0,1,0.0,0.000000
1,content_c6f503832efa4a8f,1332.0,3.0,8.767555,182.0,1,5.0,0.053706
2,content_df6a9b44e683db63,8911.0,0.0,7.015647,0.0,1,0.0,0.000000
3,content_394bee03422a2847,2754.0,0.0,6.532542,267.0,1,1.0,0.053012
4,content_b28a95c64c3b6c0f,2524.0,18.0,4.718247,310.0,1,0.0,0.000000


## Feature Justifications (Decision Moment: March 31, 2026)

1. **`imp_past_90`**: Knowable at the decision moment because it aggregates only impressions that occurred before midnight on March 31.
2. **`clicks_past_90`**: Knowable at the decision moment because past clicks are logged daily in the warehouse prior to the prediction window.
3. **`avg_pos_past_90`**: Knowable at the decision moment because historical ranking positions are locked and immutable.
4. **`visible_queries`**: Knowable at the decision moment because it measures the page's search diversity up to the cutoff date.
5. **`rare_share`**: Knowable at the decision moment because past anonymity ratios are already calculated in the query table.

In [6]:
## 4) The Leakage Trap Lesson
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. THE TRAP EXPERIMENT
# We include 'future_impressions' in our features. The model will "cheat".
trap_features = ['imp_past_90', 'clicks_past_90', 'avg_pos_past_90', 'visible_queries', 'rare_share', 'future_impressions']

X_trap = df_model[trap_features]
y = df_model['is_declining']

X_train, X_test, y_train, y_test = train_test_split(X_trap, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42, n_jobs=-1)

model.fit(X_train, y_train)
trap_score = accuracy_score(y_test, model.predict(X_test))
print(f"🚨 TRAP SCORE (With Leakage): {trap_score:.3f} (Suspiciously high because the model knows the future!)")

# 2. THE HONEST EXPERIMENT
# We delete the future column and keep only the 5 features knowable at the decision moment.
honest_features = ['imp_past_90', 'clicks_past_90', 'avg_pos_past_90', 'visible_queries', 'rare_share']
X_honest = df_model[honest_features]

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, test_size=0.2, random_state=42)

model.fit(X_train_h, y_train_h)
honest_score = accuracy_score(y_test_h, model.predict(X_test_h))
print(f"✅ HONEST SCORE (Leakage Removed): {honest_score:.3f} (Realistic real-world performance)")

print("\n## 5) Limitation of this slice:")
print("One limitation of this slice is that by excluding pages with <100 impressions, our model will be entirely blind to new content (cold starts) and will only be useful for predicting trends on established pages.")

🚨 TRAP SCORE (With Leakage): 0.842 (Suspiciously high because the model knows the future!)


✅ HONEST SCORE (Leakage Removed): 0.719 (Realistic real-world performance)

## 5) Limitation of this slice:
One limitation of this slice is that by excluding pages with <100 impressions, our model will be entirely blind to new content (cold starts) and will only be useful for predicting trends on established pages.
